# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [31]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:8] + "..." if hf_token else "NOT FOUND")

Token loaded: hf_DCVCv...


In [32]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Tables defined.")

Tables defined.


In [33]:
con.sql(f"SELECT * FROM {TABLES['dim_clients']} LIMIT 5").show()

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

In [34]:
con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 5").show()

┌─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬──────────────────────┬──────────────────────┬─────────────────┬───────────────┬─────────────┬───────────────────┬────────┬───────────────┬───────────┬────────────────┬──────────────────────┬─────────────────────────┬────────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │     keyword_hash_id      │     url_hash_id      │ keyword_char_count │ keyword_token_count │ url_char_count │ content_created_date │ content_updated_date │  content_type   │ search_volume │ competition │ competition_level │  cpc   │  main_intent  │ backlinks │ category_count │ keyword_created_date │      provider_used      │       model_used       │ char_count │ word_count │ last_optimized_date │ optimization_eligible_date │ is_p

In [35]:
#i mistakenly used the csv file first so all the code blocks have my answres with refernce to the csv first and then with reference to the huggingface dataset
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")
df.columns.tolist()
#one row = one page. There are no explicit dates/time frames provided in the csv file. Instead the data available is relative (since 90 days or 30 days) as is seen by these columns



['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [36]:
print((df["impressions_last_30d"] == df["impressions_prev_30d"]).mean())
#however, it is important to note the ovelap between the entries in impressions_last_30d and impressions_prev_30d is only around 5.3 percent. this shows that both these columns are not duplicates and hence the data available was recorded at different times. These are genuinely separate windows, not just relabeled duplicates.

0.05313333333333333


1. Unit of analysis + time window: One row = one what, over which dates?

One row = one content item (page) on one specific date. This is the grain of fact_content_daily_performance — daily × client × content — unlike the starter CSV, where each row was a pre-aggregated 90-day summary per page. Our verification and feature-building slice uses month=2026-03 (a mid-panel month with real data before and after it), while our label compares March against February.
3. Time window:

month=2026-03 for verification queries and feature-building, deliberately not the final month (June 2026) — using the last month would risk leakage, since there'd be no true "future" data left to validate against, and the last month is the natural outcome window for any past→future label.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [37]:
df["computed_pct_check"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
df[["trend_pct", "computed_pct_check"]].head(10)
df["provider_used"].unique()
df["model_used"].unique()
df["provider_used"].isna().mean()
df["ai_traffic_pct"].isna().mean()
(df["ai_traffic_pct"] == 0).mean()

np.float64(0.9356666666666666)

Feature — real, observed signals knowable before the outcome: content_age_days, days_since_last_update, search_volume, competition, cpc, word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d/prev_30d, clicks_last_30d/prev_30d, sessions_last_30d/prev_30d, ctr, avg_position, engagement_rate, scroll_rate.

Label — trend_direction (the source of is_declining_label, our proxy target).

Context — content_id, client_id (pseudonymized identifiers, no predictive pattern); age_tier, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier (bucketed duplicates of raw numeric columns already used as features — redundant, kept only for human-readable reason codes).

Excluded —

trend_pct: verified numerically (via (last_30d - prev_30d)/prev_30d * 100) to be the exact value trend_direction is derived from. Using it would be leakage — the model would learn to copy the label instead of finding real signal.
provider_used, model_used: 71.5% missing, and even where present, risk acting as a confound — any correlation between AI model and decline may really reflect when that model was adopted, not genuine content quality.
ai_traffic_pct: no missing values, but 93.56% of rows are exactly 0, meaning very little real signal for the vast majority of pages — consistent with the lane guide's warning that AI-session data is generally sparse. Excluded rather than treated as a normal feature.

from hugging face dataset:
2. Fields: which table(s)?

fact_content_daily_performance (or fact_daily_sample for quick testing) for the daily traffic/click/position signals, joined to dim_content for static page-level attributes (word_count, content_type, search_volume, etc.) via content_hash_id. dim_clients may also be used for the availability check (has_gsc_access, has_ga4_access boolean flags).

4. Target or proxy:

A proxy label — "declining" — defined by comparing a page's traffic (e.g. impressions) in March 2026 against February 2026. If March impressions are meaningfully lower than February's (e.g. a chosen % drop threshold), the page is labeled declining (1), otherwise not declining (0). This mirrors trend_direction from the starter CSV, but computed ourselves from two real, verifiable months of daily data.

5. What we deliberately exclude:

Any data from March itself (or later) must never be used as a feature — only as the label — since March is the outcome we're trying to predict. Including March's own traffic numbers as a feature would be leakage, the same trap as trend_pct in the starter CSV, just reshaped around real calendar time instead of a pre-built column.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [38]:
# Claim 1: one row = one page (grain check)
print("Shape:", df.shape)
print("Unique content_id count:", df["content_id"].nunique())
# If nunique == number of rows, confirms one row per page, no duplicates

# Claim 2: last_30d and prev_30d are genuinely separate windows, not duplicates
overlap_rate = (df["impressions_last_30d"] == df["impressions_prev_30d"]).mean()
print("Fraction identical between last_30d and prev_30d:", overlap_rate)
# ~5.3% identical -> genuinely separate windows, not a duplicated column

# Claim 3: trend_pct is derived from last_30d/prev_30d, and trend_direction from trend_pct -> leakage
df["computed_pct_check"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
print(df[["trend_pct", "computed_pct_check"]].head(10))
# Near-identical values confirm trend_pct's formula, and therefore why it must be excluded

# Claim 4: provider_used / model_used are heavily missing
print("provider_used missing rate:", df["provider_used"].isna().mean())
print("model_used unique values:", df["model_used"].unique())

# Claim 5: ai_traffic_pct has no NaNs but is mostly zero
print("ai_traffic_pct missing rate:", df["ai_traffic_pct"].isna().mean())
print("ai_traffic_pct == 0 rate:", (df["ai_traffic_pct"] == 0).mean())

# Claim 6: avg_position uses 0 as a sentinel for "no data," distorting the mean if left in
print("Rows with avg_position == 0:", (df["avg_position"] == 0).sum())
print("Mean avg_position including zeros:", df["avg_position"].mean())
print("Mean avg_position excluding zeros:", df[df["avg_position"] > 0]["avg_position"].mean())

Shape: (30000, 45)
Unique content_id count: 30000
Fraction identical between last_30d and prev_30d: 0.05313333333333333
   trend_pct  computed_pct_check
0      -41.4          -41.438703
1      -57.7          -57.717667
2      -60.9          -60.880276
3      -13.8          -13.789824
4      -34.7          -34.733416
5      -38.9          -38.850347
6      -92.3          -92.307692
7        0.6            0.632911
8      -58.8          -58.808215
9      -29.2          -29.213483
provider_used missing rate: 0.7146
model_used unique values: ['gemini-2.5-flash' 'gemini-3-flash-preview' nan 'gpt-4o-mini' 'unknown'
 'gpt-5-mini']
ai_traffic_pct missing rate: 0.0
ai_traffic_pct == 0 rate: 0.9356666666666666
Rows with avg_position == 0: 1205
Mean avg_position including zeros: 16.342380000000002
Mean avg_position excluding zeros: 17.026268449383576


In [39]:
# Claim: one row = one page (content_hash_id) on one specific date (report_date).
# To verify this, we group March rows by (content_hash_id, report_date) pairs,
# and check whether any pair appears MORE THAN ONCE — which would mean the
# grain is violated (duplicate rows for the same page on the same day).
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

# If grain truly holds, this should be an EMPTY dataframe (zero duplicate pairs).
print("Rows violating grain (should be 0 if grain holds):", len(grain_check))
#What this does, in plain terms:
#it groups all March rows by (page, date) pairs, counts how many rows exist for each pair, and only keeps groups where the count is more than 1 (meaning a duplicate — same page, same day, appearing twice, which shouldn't happen if the grain is truly one-row-per-page-per-day). If the grain is clean, this query should return zero rows — an empty result is your proof.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating grain (should be 0 if grain holds): 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

(I ask claude to ask me questions to get to the answer and it then refines or corrects my answers as needed. So the text below and other texts are generated by claude but have my content and ideas)

This starter dataset has several limits that shape what we can honestly claim:

Unbalanced/unknown history: This is a single 30,000-row anonymized slice. We don't know how many distinct clients it represents or how their history depth varies (the full warehouse has 104 clients with wildly different tracking start dates). Findings here should not be generalized to "all FlyRank clients" without checking the fuller warehouse data.

No calendar dates, only relative windows: The CSV has no explicit snapshot or report date — only relative windows (_90d, last_30d/prev_30d, days_since_last_update). This means we cannot verify whether a future "predict decline in the next 30 days" model's feature window and target window would truly avoid overlapping in time; we'd need explicit dates (available in the warehouse's report_date column) to guarantee that safely.

Sparse AI-traffic data: 93.6% of rows have ai_traffic_pct = 0. We can say most of the traffic in this dataset isn't AI-driven, but we cannot confidently claim individual pages "get no AI traffic" — the signal is simply too thin to support page-level claims.

Confounded/missing provider data: provider_used/model_used are 71.5% missing, and the missingness itself may not be random (e.g., older pages may predate certain AI tools entirely). Any comparison across AI models would likely reflect when content was made, not a genuine causal effect of the model used — so this field was excluded rather than used to support any claim.

Sentinel values limit coverage: 1,205 rows (4%) have avg_position = 0, meaning no ranking data — not a real rank of zero. Any analysis using avg_position must exclude these rows, meaning conclusions apply only to "pages with ranking data," not the full page inventory.

Correlation, not causation: Everything in this dataset is observational. We can find that certain signals (staleness, low CTR, etc.) correlate with decline, but we cannot claim any of them cause decline, and we cannot claim that reviewing a flagged page will cause it to recover — that would require an experiment or causal design, which this dataset doesn't support.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.